# Pydantic v2 Field Cheatsheet (All Possible Variants) 



```
from typing import Optional, List, Union
from pydantic import BaseModel, Field
from datetime import datetime
from typing_extensions import Literal
from enum import Enum

class EnumType(str, Enum):
    a = "a"
    b = "b"

class LLMExample(BaseModel):
    # -------- Primitive types --------
    x_int: int                                   # বাধ্যতামূলক integer, input না দিলে ValidationError
    x_int_def: int = 5                           # optional in input, missing দিলে default 5 নেবে
    x_int_field: int = Field(default=5)          # same as previous, explicit default + constraints দিতে পারবে
    x_opt_int_hint: Optional[int]                # type hint Optional, কিন্তু default নেই, missing দিলে error
    x_opt_int_none: Optional[int] = None         # truly optional, missing/None/JSON null accepted
    x_opt_int_def: Optional[int] = 5             # optional with default 5, missing → 5, None invalid unless explicitly given
    x_int_range: int = Field(..., ge=0, le=100)  # required, numeric constraint আছে, 0 <= value <= 100
    x_str_meta: str = Field(..., min_length=2, description="Candidate full name")  # required string, LLM/docs friendly description
    x_opt_str: Optional[str] = Field(None, min_length=1, max_length=20)           # optional string, constraints enforced যদি value দেওয়া হয়

    # -------- List types --------
    x_list: List[int] = Field(..., min_length=1)              # required list, কমপক্ষে 1 element থাকতে হবে
    x_list_def: List[int] = Field(default_factory=list)       # optional list, missing দিলে empty list auto তৈরি হবে

    # -------- Literal & Enum --------
    x_lit: Literal["a", "b", "c"]                             # শুধু এই value গুলো valid, hallucination stopper
    x_opt_lit: Optional[Literal["a", "b"]] = None             # optional literal, missing বা None accepted
    x_enum: EnumType                                          # required enum, predefined constant values
    x_opt_enum: Optional[EnumType] = None                     # optional enum, missing/None allowed

    # -------- Union --------
    x_union: Union[int, str]                                   # required, value হতে পারে int বা str
    x_opt_union: Optional[Union[int, str]] = None              # optional union, missing/None accepted, otherwise int/str

    # -------- Date / datetime --------
    x_dt: datetime = Field(default_factory=datetime.utcnow)   # auto timestamp, missing দিলে current datetime auto assign হবে
```

| Feature             | Covered |
| ------------------- | ------- |
| Required field      | ✅       |
| Optional field      | ✅       |
| Default value       | ✅       |
| Literal enforcement | ✅       |
| Enum usage          | ✅       |
| List + constraints  | ✅       |
| List of Literal     | ✅       |
| Union types         | ✅       |
| Email validation    | ✅       |
| URL validation      | ✅       |
| Date / datetime     | ✅       |
| Metadata dict       | ✅       |
| LLM-safe extraction | ✅       |
| DB/API ready        | ✅       |


# 1️⃣ Primitive types
| Field | Example                                      | Bangla explanation                                                      |
| ----- | -------------------------------------------- | ----------------------------------------------------------------------- |
| int   | `id: int`                                    | Required integer। কোনো default দিলে Optional হতে পারে।                  |
| str   | `name: str = Field(..., min_length=2)`       | Required string। `Field` দিয়ে extra constraint ও description দিতে পারো। |
| float | `confidence: float = Field(..., ge=0, le=1)` | Required float। value range enforce করার জন্য Field ব্যবহার।            |
| bool  | `is_remote: bool`                            | True/False। Mostly required unless default।                             |


# 2️⃣ Optional vs Required
- x:int-------------------> required
- x: Optional[int] = None → allows missing, None, or JSON null

| Type     | Example                        | Bangla                                            |
| -------- | ------------------------------ | ------------------------------------------------- |
| Required | `salary: int`                  | Input dict-এ field না থাকলে error।                |
| Optional | `salary: Optional[int] = None` | Input dict-এ field না থাকলেও চলে। Default = None। |


# 3️⃣ Field(...)
- যখন তোমার field required + extra constraints/metadata লাগবে
- না দিলে error
- Example:

```name: str = Field(..., min_length=2, description="Candidate full name")```

- Ellipsis (...) = required
- min_length, ge, le, pattern, description = rules & hint for LLM

### When to use Field?
- যদি তোমার type শুধু simple type না, constraint লাগবে
- যদি description/dataset/LLM hint দিতে চাও
- যদি default value না থাকে


# 4️⃣ Default value

- ```notice_period_days: int = Field(default=30, ge=0, le=120)```
- ```notice_period_days: int = Field(30, ge=0, le=120)```
- Default value দিলে field optional in input।
- Constraint এখনও enforce হবে।
- Field(default=...) or simple x: int = 30 works.


# 5️⃣ Literal (strict allowed values)
```role: Literal["ai", "ml", "backend"]```
- শুধুমাত্র এই value গুলোই valid
- LLM hallucination block করতে perfect
- Optional হলে: Optional[Literal[...]] = None

# 6️⃣ List types
```
skills: List[str] = Field(..., min_length=1)
preferred_locations: List[Literal["dhaka", "chittagong", "remote"]] = Field(default_factory=list)
```

- min_length (v2) → list must have at least N elements
- default_factory=list → যদি input না আসে, empty list auto তৈরি হয়


# 7️⃣ Union
```employee_code: Union[int, str]```
- Input হতে পারে multiple types
- Legacy data / LLM flexible extraction এর জন্য useful

# 8️⃣ Enum
```
class Seniority(str, Enum):
    junior = "junior"
    mid = "mid"
    senior = "senior"

seniority: Seniority
```

- Literal এর মতো strict, তবে code-wide reuse possible
- DB/API contracts friendly

# 9️⃣ Email / pattern validation
```email: Optional[str] = Field(None, pattern=r".+@.+\..+")```
- Pydantic v2 → regex gone, এখন pattern
- Optional → default None → missing input allowed

# 10️⃣ Date / datetime
```
available_from: Optional[date] = None
last_updated: datetime = Field(default_factory=datetime.utcnow)
```

- Optional date → missing allowed
- Default_factory → auto timestamp

# 11️⃣ Dict / metadata
```metadata: Optional[Dict[str, str]] = None```
- Flexible key-value storage
- Optional → safe for missing LLM output


# Bangla cheat-sheet: Rules of thumb
| Rule                 | Bangla explanation                                  |
| -------------------- | --------------------------------------------------- |
| `type` only          | Field simple, required, no extra rules              |
| `Field(...)`         | Extra constraint, metadata, description, required   |
| `Optional[T] = None` | Field missing allowed, safe for LLM                 |
| `Literal[...]`       | Only allowed values, hallucination stopper          |
| `List[T]`            | min_length constraint + default_factory if optional |
| `Union[A,B]`         | Flexible input type (int or str etc)                |
| `Enum`               | System-wide constant, reusable across codebase      |
| `pattern`            | Regex validation in v2                              |


In [ ]:
from pydantic import BaseModel, Field, EmailStr, HttpUrl
from typing import List, Optional, Literal, Union, Dict
from datetime import datetime, date
from enum import Enum



class Seniority(str, Enum):
    junior = "junior"
    mid = "mid"
    senior = "senior"

class FullExample(BaseModel):
    id: int
    name: str = Field(..., min_length=2)

    role: Literal["ai", "ml", "backend"]

    salary: Optional[int] = Field(None, ge=20000, le=500000)

    skills: List[str] = Field(..., min_length=1)

    is_remote: bool

    confidence: float = Field(..., ge=0.0, le=1.0)

    seniority: Seniority

    employee_code: Union[int, str]

    email: Optional[str] = Field(
        None,
        pattern=r".+@.+\..+"
    )

    portfolio_url: Optional[str] = None

    available_from: Optional[date] = None

    last_updated: datetime = Field(default_factory=datetime.utcnow)

    # 🔥 FIXED
    metadata: Optional[Dict[str, str]] = None

    preferred_locations: List[
        Literal["dhaka", "chittagong", "remote"]
    ] = Field(default_factory=list)

    notice_period_days: int = Field(30, ge=0, le=120)

    employment_type: Optional[
        Literal["full_time", "contract", "intern"]
    ] = None







llm_response = {
    "id": 1,
    "name": "Al Amin",
    "role": "ai",
    "salary": 80000,
    "skills": ["Python", "ML"],
    "is_remote": True,
    "confidence": 0.92,
    "seniority": "mid",
    "employee_code": "EMP-102",
    "email": "alamin@example.com",
    "portfolio_url": "https://github.com/example",
    "preferred_locations": ["dhaka", "remote"],
    "employment_type": "full_time"
}

obj = FullExample(**llm_response)

